# K5 s2024 vs LaGoVAD (released ckpt), zero-shot **MSAD test** — `KIM_RUN.md` §18

Rule written before any K5 number: `core/docs/final/KIM_RUN.md` §18. **Nothing trains.** Upload branch `v_final_las_vad` to
`Thesis-Final/kat-vad` first. Videos: the user's shortcut `Thesis-Final/data/MSAD Videos blurred` (official blurred release).

| step | what | GPU | estimate |
|---|---|---|---|
| 1-4 | setup, deps, helpers, preflight (K5 J10 + config, inputs exist) | no | 2 min |
| 5 | copy the 240 test mp4 shortcut -> VM disk | no | 5-10 min |
| 6 | **census + gate M2** (each id once, `ceil(decoded/8) == rows(MSAD_ncc)`) | no | 5-10 min |
| 7 | **gate M1**: CLIP `_ncc` re-extracted from 12 mp4 == `MSAD_ncc` | yes | 2 min |
| 8 | decode mp4 -> JPEG (VM disk) | no | 10-20 min |
| 9 | VideoMAE V2-S s8, frame step 3 -> `Thesis-Final/cache/video/…/MSAD-full_s8_squash` | **yes** | 20-40 min |
| 10 | bake A2 (no CRN) with **Nexar's** statistics -> `Thesis-Final/cache/v2/A2_S/MSAD-full_s8_nexar` | no | 2 min |
| 11 | `core.evaluate` LaGoVAD + **gate M0** (micro 0.8949 ± 1e-3) | light | 3 min |
| 12 | `core.evaluate` K5 s2024 | light | 3 min |
| 13 | `core.tools.k5_msad_report` -> **paste `REPORTS/k5_msad_report.md` to Claude** | no | 1 min |

Every step skips what is already on Drive. One seed: **PRELIMINARY (n = 1), not matched.**

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import json
import os
import sys
from pathlib import Path

DRIVE    = '/content/drive/MyDrive/Thesis'        # read-only: data/MSAD, cache/clip/MSAD_ncc, ckpts/best.ckpt
DRIVE_V2 = '/content/drive/MyDrive/Thesis-V2'     # read-only: A2 Nexar statistics, Nexar V2-S manifest
DRIVE_F  = '/content/drive/MyDrive/Thesis-Final'  # code, K5's run, the MSAD shortcut, every output
os.environ['PROJECT_ROOT']       = DRIVE_F
os.environ['KATVAD_DATA_ROOT']   = f'{DRIVE}/data'
os.environ['KATVAD_CACHE_ROOT']  = f'{DRIVE_F}/cache'
os.environ['KATVAD_CKPT_ROOT']   = f'{DRIVE}/ckpts'
os.environ['KATVAD_OUTPUT_ROOT'] = f'{DRIVE_F}/outputs'

os.environ['REPO'] = f'{DRIVE_F}/kat-vad'
REPO = Path(os.environ['REPO'])
assert (REPO / 'core' / 'constants.py').is_file(), f'no code at {REPO} -- upload branch v_final_las_vad there'
assert (REPO / 'core/tools/k5_msad_report.py').is_file(), 'this upload has no k5_msad_report -- re-upload kat-vad/'
os.environ['PYTHONPATH'] = str(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
for name in [m for m in sys.modules if m == 'core' or m.startswith('core.')]:
    del sys.modules[name]              # constants reads KATVAD_* at import time
from core import constants  # noqa: E402

assert constants.REPO_ROOT == REPO, f'imported core from {constants.REPO_ROOT}, expected {REPO}'
from core.tools import msad_zs as _msad_zs  # noqa: E402
assert hasattr(_msad_zs, 'census_current'), 'this upload predates the census re-run fix -- re-upload kat-vad/'

ARM, SEED = constants.KIM_MAIN_ARM, constants.KIM_SEED     # K5, 2024
LAG, LAG_RUN = constants.ZS_LAGOVAD_ARM, constants.ZS_LAGOVAD_RUN
DSET     = constants.MSAD_FULL_DATASET                      # 'MSAD-full' (full_gate_a's data.dataset)
STRIDE   = constants.FRAME_STRIDE                           # 8
VSTEP    = constants.VIDEOMAE_CLIP_FRAME_STEP               # 3 (Nexar geometry)
FPS      = constants.MSAD_ZS_ASSUMED_FPS                    # 30
ENC      = constants.V5_ENCODER                             # V2-S

VIDEOS   = Path(DRIVE_F) / 'data' / constants.MSAD_ZS_VIDEOS_DIRNAME   # the user's shortcut (read-only)
DATA     = Path(DRIVE) / 'data' / 'MSAD'                                # labels + defs, as full_gate_a
CLIP     = Path(DRIVE) / 'cache' / 'clip' / 'MSAD_ncc'                  # LaGoVAD's input, as full_gate_a
CKPT_LAG = Path(DRIVE) / 'ckpts' / 'best.ckpt'
A2_STATS = Path(DRIVE_V2) / 'cache' / 'v2' / 'A2_S' / 'Nexar_s8'        # K5's training bake: Nexar statistics
NEXAR_V2S = Path(DRIVE_V2) / 'cache' / 'video' / ENC / f'{constants.NEXAR_VIDEO_S1_DATASET}_squash'

KIM      = constants.OUTPUT_ROOT / constants.KIM_OUTPUT_DIRNAME
RUN      = KIM / ARM / f's{SEED}'
CKPT     = RUN / 'checkpoint_last.pt'
V2S      = constants.CACHE_ROOT / 'video' / ENC / f'{DSET}_s{STRIDE}_squash'
BAKE     = constants.CACHE_ROOT / 'v2' / 'A2_S' / f'{DSET}_s{STRIDE}_nexar'
ROOT     = constants.OUTPUT_ROOT / constants.MSAD_ZS_OUTPUT_DIRNAME     # Thesis-Final/outputs/kim_msad
EVAL     = ROOT / 'eval'
REPORTS  = ROOT / 'REPORTS'
CENSUS   = REPORTS / constants.MSAD_ZS_CENSUS_FILENAME
IDENTITY = REPORTS / constants.MSAD_ZS_IDENTITY_FILENAME
LOCAL    = Path('/content/k5_msad')
L_VIDEOS = LOCAL / 'videos'                                             # VM copy of the 240 test mp4
FRAMES   = LOCAL / 'frames'                                             # VM disk, never Drive
for p in (REPORTS, LOCAL):
    p.mkdir(parents=True, exist_ok=True)

for name, path in (('MSAD shortcut', VIDEOS), ('MSAD labels', DATA / constants.FRAME_LABELS_TEST_FILENAME),
                   ('MSAD defs', DATA / constants.DEFS_FILENAME), ('MSAD_ncc', CLIP), ('best.ckpt', CKPT_LAG),
                   ('A2 Nexar stats', A2_STATS / constants.V2_INPUT_MANIFEST_FILENAME),
                   ('Nexar V2-S manifest', NEXAR_V2S / constants.VIDEO_MANIFEST_FILENAME),
                   ('K5 checkpoint', CKPT), ('K5 verified.json', RUN / 'verified.json'),
                   ('K5 metrics', RUN / 'metrics.jsonl'), ('K5 config', RUN / 'config.yaml')):
    assert path.exists(), f'missing {name}: {path}'
    print(f'  {name:20s} {path}')
LABELS = json.loads((DATA / constants.FRAME_LABELS_TEST_FILENAME).read_text())
IDS = sorted(LABELS)
assert len(IDS) == constants.MSAD_ZS_TEST_VIDEOS, len(IDS)
print(f'{len(IDS)} test videos; caches -> {V2S}, {BAKE}; outputs -> {ROOT}')

In [ ]:
%%bash
pip install -q "transformers==4.56.*" av einops faiss-cpu scikit-learn openpyxl
python -c "import av, einops, faiss, sklearn; print('deps OK')"
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'NO GPU -- steps 7, 9 need one'
df -h /content | tail -1

In [ ]:
import shutil
import subprocess
import time

import numpy as np
import yaml

ENV = dict(os.environ, USE_TF='0', TF_CPP_MIN_LOG_LEVEL='2')   # transformers must not import TF


def run(cmd, log=None):
    '''Stream a child's output here (and to ``log``); raise with its tail on failure.'''
    tail = []
    sink = open(log, 'a') if log else None
    try:
        with subprocess.Popen([str(c) for c in cmd], cwd=str(REPO), env=ENV, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1) as proc:
            for line in proc.stdout:
                tail = (tail + [line])[-40:]
                if sink:
                    sink.write(line)
                print(line, end='')
    finally:
        if sink:
            sink.close()
    if proc.returncode:
        print('\n' + '=' * 30 + f'\nFAILED (exit {proc.returncode}) -- paste these lines to Claude:\n' + ''.join(tail))
        raise RuntimeError(f'exit {proc.returncode}: {" ".join(str(c) for c in cmd[:4])} ...')


def publish(src, dst):
    '''VM-local result -> Drive ATOMICALLY: ``<dst>.partial``, fsync, rename (C10).'''
    if src.is_dir():
        for f in [p for p in src.rglob('*') if p.is_file()]:
            publish(f, dst / f.relative_to(src))
        return
    dst.parent.mkdir(parents=True, exist_ok=True)
    tmp = dst.with_name(dst.name + '.partial')
    shutil.copy2(src, tmp)
    with open(tmp, 'rb+') as fh:
        os.fsync(fh.fileno())
    os.replace(tmp, dst)


def stage_ids(src, dst, ids, suffix='.npy'):
    '''Only the test ids' files of a Drive cache -> VM disk (bit-identical; C2).'''
    dst.mkdir(parents=True, exist_ok=True)
    for v in ids:
        if not (dst / f'{v}{suffix}').exists():
            shutil.copy2(src / f'{v}{suffix}', dst / f'{v}{suffix}')
    for f in src.glob('*.json'):                     # manifests travel with the arrays
        shutil.copy2(f, dst / f.name)
    return dst


def complete(d, ids):
    return d.is_dir() and all((d / f'{v}.npy').exists() for v in ids)


def rows(d, v):
    return int(np.load(d / f'{v}.npy', mmap_mode='r').shape[0])


def write_ids(path, ids):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(''.join(f'{v}\n' for v in ids))
    return path


def flush_drive():
    t0 = time.time()
    drive.flush_and_unmount()
    drive.mount('/content/drive')
    print(f'Drive flushed and remounted in {time.time() - t0:.0f}s')


ID_FILE = write_ids(LOCAL / 'msad_test_ids.txt', IDS)

# Preflight on small files only (C39): K5 J10 from the record written when the checkpoint was verified (M4).
verified = json.loads((RUN / 'verified.json').read_text())
lines = [l for l in (RUN / 'metrics.jsonl').read_text().splitlines() if l.strip()]
last_step = json.loads(lines[-1])['global_step']
assert verified['step'] == last_step, f'J10: verified step {verified["step"]} != metrics {last_step}'
assert CKPT.stat().st_size == verified['ckpt_bytes'], f'checkpoint size {CKPT.stat().st_size} != verified {verified["ckpt_bytes"]}'
cfg = yaml.safe_load((RUN / 'config.yaml').read_text())
want = {'enabled': True, 'prototypes': True, 'edge_mask': True}
assert {k: cfg['kim'][k] for k in want} == want, f'not K5: {cfg["kim"]}'
assert cfg['dvs']['theta'] == 1.0 and cfg['train']['seed'] == SEED and cfg['kip']['enabled'] is False, 'not K5 s2024'
print(f'{ARM} s{SEED}: step {last_step} (J10 OK), kim {want}, DVS off')

## 5. Copy the 240 test mp4 from the shortcut's **official folders** (`MSAD_anomaly/`, `MSAD_normal_testing/`; never `VALU_MSAD_subset_2026/`) to VM disk

In [ ]:
NEED_VIDEOS = not (CENSUS.exists() and json.loads(CENSUS.read_text()).get('source_dirs') == list(constants.MSAD_ZS_SOURCE_DIRS)
                   and json.loads(CENSUS.read_text())['summary']['pass'] and IDENTITY.exists() and complete(V2S, IDS))
if NEED_VIDEOS:
    stale = [d for d in L_VIDEOS.iterdir() if d.name not in constants.MSAD_ZS_SOURCE_DIRS] if L_VIDEOS.exists() else []
    for d in stale:                                  # a VALU copy from an earlier run: never read it
        shutil.rmtree(d)
    want_ids, t0, n = set(IDS), time.time(), 0
    # official folders only: the shortcut also holds VALU_MSAD_subset_2026, a MODIFIED copy (watermarks masked)
    for p in (q for d in constants.MSAD_ZS_SOURCE_DIRS for q in (VIDEOS / d).rglob('*.mp4')):
        if p.stem in want_ids:
            dst = L_VIDEOS / p.relative_to(VIDEOS)
            if not dst.exists():
                dst.parent.mkdir(parents=True, exist_ok=True)
                tmp = dst.with_name(dst.name + '.partial')
                shutil.copy2(p, tmp)
                os.replace(tmp, dst)
            n += 1
    print(f'{n} test mp4 on VM disk in {(time.time() - t0) / 60:.1f} min (the census checks each id is there exactly once)')
else:
    print('census, M1 and V2-S are on Drive -- nothing copied')

## 6. Census + gate M2 (`KIM_RUN.md` §18) — stops here on any failure

In [ ]:
from core.tools.msad_zs import census_current  # noqa: E402

if not (CENSUS.exists() and census_current(json.loads(CENSUS.read_text()))):
    if CENSUS.exists():
        print('the census on Drive failed or predates the VALU exclusion -- re-running it')
    local = LOCAL / constants.MSAD_ZS_CENSUS_FILENAME
    try:
        run([sys.executable, '-m', 'core.tools.msad_zs', 'census', '--videos-root', L_VIDEOS,
             '--labels', DATA / constants.FRAME_LABELS_TEST_FILENAME, '--clip-dir', CLIP,
             '--stride', STRIDE, '--out', local], log=LOCAL / 'census.log')
    finally:
        if local.exists():
            publish(local, CENSUS)
doc = json.loads(CENSUS.read_text())
print(json.dumps(doc['summary'], indent=1))
assert doc['summary']['pass'], 'MSAD census FAILED (M2) -- paste the summary above to Claude'

## 7. Gate M1: CLIP `_ncc` re-extracted straight from 12 test mp4 must equal `MSAD_ncc` (else: other videos, stop)

In [ ]:
if not IDENTITY.exists():
    m1_ids = LOCAL / 'm1_ids.txt'
    run([sys.executable, '-m', 'core.tools.msad_zs', 'pick', '--census', CENSUS, '--out', m1_ids])
    picked = m1_ids.read_text().split()
    m1_videos = LOCAL / 'm1_videos'                  # extract_clip_features --videos-dir takes no --ids-file
    m1_videos.mkdir(parents=True, exist_ok=True)
    for v in picked:
        src = L_VIDEOS / doc['videos'][v]['video']
        if not (m1_videos / src.name).exists():
            shutil.copy2(src, m1_videos / src.name)
    run([sys.executable, '-m', 'core.tools.extract_clip_features', '--videos-dir', m1_videos, '--dataset', DSET,
         '--stride', STRIDE, '--no-center-crop', '--device', 'auto', '--output-dir', LOCAL / 'm1_clip'],
        log=LOCAL / 'm1.log')
    local = LOCAL / constants.MSAD_ZS_IDENTITY_FILENAME
    try:
        run([sys.executable, '-m', 'core.tools.msad_zs', 'identity', '--ref', CLIP, '--new', LOCAL / 'm1_clip',
             '--ids-file', m1_ids, '--out', local])
    finally:
        if local.exists():
            publish(local, IDENTITY)
ident = json.loads(IDENTITY.read_text())
print({k: ident[k] for k in ('videos', 'max_abs', 'atol', 'pass')})
assert ident['pass'], 'M1 FAILED: the shortcut videos are not MSAD_ncc\'s -- paste this to Claude, do not go on'

## 8. Decode mp4 -> JPEG folders (VM disk) — only if V2-S is incomplete

In [ ]:
if not complete(V2S, IDS):
    t0 = time.time()
    run([sys.executable, '-m', 'core.tools.msad_zs', 'decode', '--videos-root', L_VIDEOS, '--census', CENSUS,
         '--out-dir', FRAMES], log=LOCAL / 'decode.log')
    print(f'decoded in {(time.time() - t0) / 60:.1f} min')
else:
    print('V2-S complete -- not decoded')

## 9. VideoMAE V2-S at s8, frame step 3, 30 fps (Nexar's geometry) -> `Thesis-Final/cache/video/…/MSAD-full_s8_squash`

In [ ]:
import torch

if complete(V2S, IDS):
    print(f'{V2S} complete')
else:
    assert torch.cuda.is_available(), 'no GPU'
    local, t0 = LOCAL / 'v2s', time.time()
    run([sys.executable, '-m', 'core.tools.extract_video_features', '--frames-dir', FRAMES, '--ids-file', ID_FILE,
         '--dataset', DSET, '--encoder', ENC, '--stride', STRIDE, '--clip-frame-step', VSTEP,
         '--assumed-fps', FPS, '--batch-size', 32, '--device', 'auto', '--output-dir', local], log=LOCAL / 'v2s.log')
    print(f'V2-S in {(time.time() - t0) / 60:.1f} min')
    publish(local, V2S)
    flush_drive()
mine = json.loads((V2S / constants.VIDEO_MANIFEST_FILENAME).read_text())
ref = json.loads((NEXAR_V2S / constants.VIDEO_MANIFEST_FILENAME).read_text())
diff = sorted(k for k in set(mine) | set(ref) if mine.get(k) != ref.get(k))
assert set(diff) <= {'stride'} and mine['stride'] == STRIDE, f'MSAD V2-S manifest differs from Nexar\'s on {diff} (MS3)'
L_CLIP = stage_ids(CLIP, LOCAL / 'stage' / 'clip', IDS)
L_V2S = stage_ids(V2S, LOCAL / 'stage' / 'v2s', IDS)
bad = [v for v in IDS if rows(L_V2S, v) != rows(L_CLIP, v)]
assert not bad, f'V2-S rows != MSAD_ncc rows on {bad[:5]} (M3, C13)'
print(f'V2-S OK: manifest == Nexar\'s except {diff}; rows == MSAD_ncc rows on {len(IDS)} videos')

## 10. Bake A2 (no CRN, V2-S) with **Nexar's** statistics -> `Thesis-Final/cache/v2/A2_S/MSAD-full_s8_nexar` (CPU)

In [ ]:
man_path = BAKE / constants.V2_INPUT_MANIFEST_FILENAME
if not (man_path.exists() and json.loads(man_path.read_text())['baked_ids'] == len(IDS)):
    local = LOCAL / 'bake'
    stats = LOCAL / 'stats_a2'
    if not stats.exists():
        shutil.copytree(A2_STATS, stats)
    run([sys.executable, '-m', 'core.tools.build_v2_inputs', 'apply', '--stats-dir', stats, '--clip-dir', L_CLIP,
         '--video-dir', L_V2S, '--ids-file', ID_FILE, '--stride', 1, '--out-dir', local], log=LOCAL / 'bake.log')
    publish(local, BAKE)
    flush_drive()
m = json.loads(man_path.read_text())
nexar_sha = json.loads((A2_STATS / constants.V2_INPUT_MANIFEST_FILENAME).read_text())['train_ids_sha1']
assert (m['crn'], m['motion']) == (constants.V2_OFF, ENC) and m['train_ids_sha1'] == nexar_sha, m
L_BAKE = stage_ids(BAKE, LOCAL / 'stage' / 'bake', IDS)
bad = [v for v in IDS if rows(L_BAKE, v) != rows(L_CLIP, v)]
assert not bad, f'bake rows != CLIP rows on {bad[:5]} (M3)'
print(f'bake OK: A2 (no CRN, {ENC}), Nexar statistics {nexar_sha[:12]}, {len(IDS)} videos')

## 11-12. Score LaGoVAD (gate M0) and K5 with `core.evaluate` (whole video, s8, raw scores, MSAD's own definitions)

In [ ]:
L_DATA = LOCAL / 'stage' / 'data'                    # json only -- never a video folder
L_DATA.mkdir(parents=True, exist_ok=True)
for f in DATA.iterdir():
    if f.is_file() and f.suffix in ('.json', '.txt'):
        shutil.copy2(f, L_DATA / f.name)

common = ['--set', f'data.dataset={DSET}', '--data-dir', L_DATA, '--score-norm', 'auto', '--save-scores']
JOBS = {
    LAG: (LAG_RUN, ['--baseline-ckpt', CKPT_LAG, *[x for o in constants.RELEASED_CKPT_OVERRIDES for x in ('--set', o)],
                    '--clip-dir', L_CLIP]),
    ARM: (f's{SEED}', ['--ckpt', CKPT, '--clip-dir', L_BAKE]),
}
for row, (name, args) in JOBS.items():
    out = EVAL / row / name
    if (out / 'results.json').exists():
        print(f'msad {row}: already read')
    else:
        local = LOCAL / 'eval' / row / name
        shutil.rmtree(local, ignore_errors=True)
        run([sys.executable, '-m', 'core.evaluate', *args, *common, '--output-dir', local], log=LOCAL / f'eval_{row}.log')
        publish(local, out)
    r = json.loads((out / 'results.json').read_text())
    assert r['score_norm'] == 'none' and r['num_videos'] == constants.MSAD_ZS_TEST_VIDEOS, r
    print(f"msad {row}: macro {r['auc_macro']:.4f} over {r['auc_macro_videos']} | micro {r['auc']:.4f} | AP {r['ap']:.4f}")
    if row == LAG:
        gap = r['auc'] - constants.MSAD_ZS_GATE_A_MICRO
        print(f'  M0: micro {r["auc"]:.4f} vs full_gate_a {constants.MSAD_ZS_GATE_A_MICRO} (gap {gap:+.5f})')
        assert abs(gap) <= constants.MSAD_ZS_GATE_A_TOL, 'M0 FAILED: released ckpt does not reproduce full_gate_a -- stop, paste this to Claude'
flush_drive()

## 13. The table -> **paste `REPORTS/k5_msad_report.md` to Claude**

In [ ]:
local = LOCAL / 'report'
run([sys.executable, '-m', 'core.tools.k5_msad_report', '--root', ROOT, '--seed', SEED, '--out-dir', local])
publish(local, REPORTS)
print((REPORTS / 'k5_msad_report.md').read_text())
drive.flush_and_unmount()          # C10: every pending Drive upload lands before the VM goes
print('Drive flushed -- paste REPORTS/k5_msad_report.md to Claude')